# 03 — Baselines & Classical ML
Walk-forward evaluation. A model earns its keep only by beating the naive baselines.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import pandas as pd
from src.db.engine import get_connection
con = get_connection(read_only=True)

In [ ]:
from src.features.build import feature_columns
from src.models.ml_models import run_walk_forward, MODEL_NAMES
from src.models.baselines import BASELINES
df = con.execute('SELECT * FROM features_weekly').fetch_df()
df['date'] = pd.to_datetime(df['date'])
feats = feature_columns(df)
results = {name: run_walk_forward(df, feats, 'label_fwd_log_ret_5d', name, min_train=60, test_size=8, step=8, embargo=1)
           for name in list(BASELINES) + MODEL_NAMES}
pd.DataFrame({k: v['metrics'] for k, v in results.items()}).T